<div style="background-color:#F3F2EE">
    <br /><br />
        <p style="text-align: center;">
            <font size="6" color='#0A1781'>
                <strong>
                    Databricks Credit Risk Lakehouse - Data Science Project
                </strong>
              </font>
        </p>
        <p style="text-align: center;">
            <font size="6" color='#C58A1E'>
                <strong>
                    Data Source and Raw Layer Preparation
                </strong>
            </font>
        </p>
        <p style="text-align: center;">
            <font size="5" color='#C58A1E'>
                <strong>
                    Apresentar o propósito do notebook e posicionar a preparação<br />da camada Raw dentro da arquitetura Lakehouse.
                </strong>
            </font>
        </p>
    <br />
</div>

<div style="background-color:#F3F2EE">
    <p style="text-align: right;">
      <font size="4" color='#444444'>
            Roberto SSoares - LfLngLrnng
      </font>
    </p>
    <p style="text-align: right;"><font size="2" color='#444444'>
        <a href="https://www.linkedin.com/in/roberto-dos-santos-soares/">in/roberto-dos-santos-soares</a><br /><a href="https://roberto-ssoares.github.io/meu-portfolio/">Portifólio: roberto-ssoares</a>
    </p>
    <p style="text-align: right;">
        <font size="4" color='#444444'>
            " [+] Faturamento [-] Custo [+] Qualidade de vida "
        </font>
        <br />
        <font size="2" color='#918e8e'>"Mestre Bruno Jardim"
        </font>
    </p>        
    <p style="text-align: right;">        
        <font size="2" color='#918e8e'>           
        </font>
    </p>
</div>

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>📌 Objetivo</strong></font>

<font size="2" color='#66666'>

>- **Ações realizadas**
    - *Definição das entidades iniciais do domínio de risco de crédito.*
    - *Geração de dados sintéticos controlados.*
    - *Criação dos arquivos brutos na camada Raw.*
    - *Documentação inicial de schema, qualidade e manifesto dos arquivos.*

>- **Justificativa técnica**
    - Antes de construir camadas Bronze, Silver e Gold, é necessário definir uma fonte de dados inicial rastreável.
    - A camada Raw preserva os arquivos originais e serve como ponto de partida para ingestão, auditoria e reprocessamento.

>- **Resultados esperados**
    - Ao final deste notebook, teremos quatro arquivos CSV brutos representando clientes, contratos, pagamentos e eventos de crédito,
        - além de documentação e evidências para atualização futura do Knowledge Graph.

---

</font></div>

In [1]:
#!uv pip install watermark -q -U
#!uv pip install tabulate -q -U

In [2]:
from datetime import date, datetime, timedelta
from pathlib import Path
import random
import string

import numpy as np
import pandas as pd

In [3]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "RobertoSSoares-LfLngLrnng"

Author: RobertoSSoares-LfLngLrnng



In [4]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

In [5]:
PROJECT_NAME = "Databricks Credit Risk Lakehouse"
NOTEBOOK_NAME = "02_data_source_and_raw_layer_preparation"
STUDY_DATE = date.today().isoformat()

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"
EXPORTS_DIR = DATA_DIR / "exports"
DOCS_DIR = BASE_DIR / "docs"

for directory in [
    RAW_DIR,
    BRONZE_DIR,
    SILVER_DIR,
    GOLD_DIR,
    EXPORTS_DIR,
    DOCS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

{
    "project": PROJECT_NAME,
    "notebook": NOTEBOOK_NAME,
    "study_date": STUDY_DATE,
    "base_dir": str(BASE_DIR),
    "raw_dir": str(RAW_DIR),
}


{'project': 'Databricks Credit Risk Lakehouse',
 'notebook': '02_data_source_and_raw_layer_preparation',
 'study_date': '2026-04-28',
 'base_dir': 'D:\\_DS-Projects\\Data-Science\\databricks-credit-risk-lakehouse',
 'raw_dir': 'D:\\_DS-Projects\\Data-Science\\databricks-credit-risk-lakehouse\\data\\raw'}

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 1. Estratégia da camada Raw</strong></font>

<font size="2" color='#66666'>

>- A camada Raw representa a chegada dos arquivos originais.

>- Neste projeto, os arquivos serão sintéticos, mas controlados para simular um cenário realista de risco de crédito.

>- **Entidades iniciais**
    - `customers`: cadastro de clientes;
    - `contracts`: contratos de crédito;
    - `payments`: histórico de pagamentos;
    - `credit_events`: eventos relevantes de crédito.

>- **Princípio importante**
    A camada Raw não deve corrigir os dados.

>- Ela deve preservar os arquivos como chegaram, permitindo que as regras de limpeza, padronização e qualidade sejam aplicadas posteriormente nas camadas Bronze e Silver.

---

</font></div>

In [6]:
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

N_CUSTOMERS = 500
MIN_CONTRACTS_PER_CUSTOMER = 0
MAX_CONTRACTS_PER_CUSTOMER = 3
MAX_PAYMENT_MONTHS = 24

GENERATION_CONFIG = {
    "random_seed": RANDOM_SEED,
    "n_customers": N_CUSTOMERS,
    "min_contracts_per_customer": MIN_CONTRACTS_PER_CUSTOMER,
    "max_contracts_per_customer": MAX_CONTRACTS_PER_CUSTOMER,
    "max_payment_months": MAX_PAYMENT_MONTHS,
}

GENERATION_CONFIG

{'random_seed': 42,
 'n_customers': 500,
 'min_contracts_per_customer': 0,
 'max_contracts_per_customer': 3,
 'max_payment_months': 24}

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 2. Funções auxiliares</strong></font>

<font size="2" color='#66666'>

>- As funções abaixo serão usadas para gerar dados sintéticos reprodutíveis.
    - A ideia não é criar dados perfeitos, mas sim dados suficientemente realistas para alimentar uma esteira Raw → Bronze → Silver → Gold.

---

</font></div>

In [7]:
def random_date(start: date, end: date) -> date:
    """
    Gera uma data aleatória entre duas datas.
    """
    delta_days = (end - start).days
    return start + timedelta(days=random.randint(0, delta_days))


def make_id(prefix: str, number: int, width: int = 6) -> str:
    """
    Cria identificadores padronizados.
    """
    return f"{prefix}{number:0{width}d}"


def random_choice_with_probs(values: list, probs: list):
    """
    Escolhe um valor com base em probabilidades.
    """
    return np.random.choice(values, p=probs)


def calculate_risk_band(credit_score: int) -> str:
    """
    Classifica o cliente em uma faixa simples de risco.
    """
    if credit_score >= 720:
        return "low"
    if credit_score >= 620:
        return "medium"
    return "high"


def safe_to_csv(df: pd.DataFrame, path: Path) -> None:
    """
    Salva CSV com encoding UTF-8.
    """
    df.to_csv(path, index=False, encoding="utf-8")

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 3. Geração da entidade "customers"</strong></font>

<font size="2" color='#66666'>

>- A tabela `customers` representa o cadastro dos clientes.

>- Ela inclui dados como:
    - identificador do cliente;
    - idade;
    - renda mensal;
    - tipo de ocupação;
    - região;
    - score sintético inicial;
    - data de cadastro.

>Algumas pequenas imperfeições serão introduzidas de propósito, simulando problemas comuns em arquivos brutos.

---

</font></div>

In [8]:
employment_types = ["formal_employee", "self_employed", "public_servant", "retired", "unemployed"]
employment_probs = [0.46, 0.24, 0.12, 0.10, 0.08]

regions = ["SP", "RJ", "MG", "PR", "RS", "BA", "PE", "GO"]
region_probs = [0.36, 0.14, 0.13, 0.09, 0.08, 0.08, 0.06, 0.06]

customers_rows = []

for i in range(1, N_CUSTOMERS + 1):
    age = int(np.clip(np.random.normal(loc=41, scale=12), 18, 75))
    employment_type = random_choice_with_probs(employment_types, employment_probs)
    
    base_income = {
        "formal_employee": np.random.normal(5200, 2100),
        "self_employed": np.random.normal(6200, 3300),
        "public_servant": np.random.normal(6800, 2200),
        "retired": np.random.normal(3600, 1400),
        "unemployed": np.random.normal(1800, 900),
    }[employment_type]
    
    monthly_income = round(max(base_income, 500), 2)
    credit_score = int(np.clip(np.random.normal(loc=650, scale=95), 300, 900))
    
    customers_rows.append(
        {
            "customer_id": make_id("C", i),
            "customer_name": f"Customer {i:04d}",
            "age": age,
            "monthly_income": monthly_income,
            "employment_type": employment_type,
            "region": random_choice_with_probs(regions, region_probs),
            "credit_score_base": credit_score,
            "risk_band_source": calculate_risk_band(credit_score),
            "signup_date": random_date(date(2018, 1, 1), date(2025, 12, 31)).isoformat(),
            "source_system": "crm_core",
            "extraction_date": STUDY_DATE,
        }
    )

customers = pd.DataFrame(customers_rows)

customers.head()


,customer_id,customer_name,age,monthly_income,employment_type,region,credit_score_base,risk_band_source,signup_date,source_system,extraction_date
0,C000001,Customer 0001,46,7501.58,public_servant,SP,594,high,2025-03-04,crm_core,2026-04-28
1,C000002,Customer 0002,34,500.00,formal_employee,RJ,579,high,2019-04-02,crm_core,2026-04-28
2,C000003,Customer 0003,58,4725.87,formal_employee,SP,592,high,2018-04-13,crm_core,2026-04-28
3,C000004,Customer 0004,52,5666.21,self_employed,PR,686,medium,2021-01-31,crm_core,2026-04-28
4,C000005,Customer 0005,43,6921.51,self_employed,SP,723,low,2020-09-30,crm_core,2026-04-28


In [9]:
# Introdução controlada de pequenas imperfeições na camada Raw

customers_raw = customers.copy()

# 3% de renda mensal ausente
missing_income_idx = customers_raw.sample(frac=0.03, random_state=RANDOM_SEED).index
customers_raw.loc[missing_income_idx, "monthly_income"] = np.nan

# 1% de região inválida
invalid_region_idx = customers_raw.sample(frac=0.01, random_state=RANDOM_SEED + 1).index
customers_raw.loc[invalid_region_idx, "region"] = "UNKNOWN"

# 1 registro duplicado proposital
customers_raw = pd.concat(
    [customers_raw, customers_raw.sample(n=1, random_state=RANDOM_SEED + 2)],
    ignore_index=True,
)

customers_raw.tail()

,customer_id,customer_name,age,monthly_income,employment_type,region,credit_score_base,risk_band_source,signup_date,source_system,extraction_date
496,C000497,Customer 0497,25,1829.44,unemployed,SP,659,medium,2024-01-06,crm_core,2026-04-28
497,C000498,Customer 0498,42,5642.41,self_employed,SP,722,low,2021-09-18,crm_core,2026-04-28
498,C000499,Customer 0499,33,7091.12,formal_employee,SP,690,medium,2018-04-24,crm_core,2026-04-28
499,C000500,Customer 0500,59,500.00,self_employed,SP,866,low,2019-04-18,crm_core,2026-04-28
500,C000092,Customer 0092,53,3665.82,retired,RJ,585,high,2023-03-09,crm_core,2026-04-28


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 4. Geração da entidade "contracts"</strong></font>

<font size="2" color='#66666'>

>- A tabela `contracts` representa contratos de crédito.

>- Cada cliente pode ter nenhum, um ou mais contratos.

>- Os contratos incluem:
    - tipo de produto;
    - valor principal;
    - taxa de juros;
    - prazo;
    - data de início;
    - status;
    - relacionamento com o cliente.

---

</font></div>

In [10]:
product_types = ["personal_loan", "credit_card", "auto_loan", "payroll_loan"]
product_probs = [0.36, 0.34, 0.16, 0.14]

contract_rows = []
contract_counter = 1

for _, customer in customers.iterrows():
    n_contracts = np.random.choice(
        [0, 1, 2, 3],
        p=[0.16, 0.52, 0.24, 0.08],
    )
    
    for _ in range(n_contracts):
        product_type = random_choice_with_probs(product_types, product_probs)
        
        principal_multiplier = {
            "personal_loan": np.random.uniform(0.6, 2.8),
            "credit_card": np.random.uniform(0.2, 1.2),
            "auto_loan": np.random.uniform(3.0, 9.0),
            "payroll_loan": np.random.uniform(0.8, 3.5),
        }[product_type]
        
        principal_amount = round(customer["monthly_income"] * principal_multiplier, 2)
        
        interest_rate_monthly = {
            "personal_loan": np.random.uniform(0.025, 0.075),
            "credit_card": np.random.uniform(0.045, 0.13),
            "auto_loan": np.random.uniform(0.012, 0.035),
            "payroll_loan": np.random.uniform(0.010, 0.030),
        }[product_type]
        
        term_months = int(
            {
                "personal_loan": np.random.choice([12, 18, 24, 36, 48]),
                "credit_card": np.random.choice([12, 24]),
                "auto_loan": np.random.choice([36, 48, 60]),
                "payroll_loan": np.random.choice([24, 36, 48, 60]),
            }[product_type]
        )
        
        start_dt = random_date(date(2020, 1, 1), date(2025, 10, 31))
        
        status = random_choice_with_probs(
            ["active", "closed", "defaulted"],
            [0.70, 0.22, 0.08],
        )
        
        contract_rows.append(
            {
                "contract_id": make_id("CTR", contract_counter),
                "customer_id": customer["customer_id"],
                "product_type": product_type,
                "principal_amount": principal_amount,
                "interest_rate_monthly": round(interest_rate_monthly, 4),
                "term_months": term_months,
                "start_date": start_dt.isoformat(),
                "contract_status": status,
                "source_system": "loan_management",
                "extraction_date": STUDY_DATE,
            }
        )
        
        contract_counter += 1

contracts = pd.DataFrame(contract_rows)

contracts.head()


,contract_id,customer_id,product_type,principal_amount,interest_rate_monthly,term_months,start_date,contract_status,source_system,extraction_date
0,CTR000001,C000001,credit_card,2863.83,0.0844,24,2022-12-05,active,loan_management,2026-04-28
1,CTR000002,C000002,auto_loan,3431.72,0.0225,48,2022-01-01,closed,loan_management,2026-04-28
2,CTR000003,C000002,credit_card,232.56,0.0538,24,2022-12-23,active,loan_management,2026-04-28
3,CTR000004,C000003,credit_card,4453.18,0.0637,24,2020-06-05,closed,loan_management,2026-04-28
4,CTR000005,C000003,payroll_loan,15677.32,0.0193,24,2021-03-20,defaulted,loan_management,2026-04-28


In [11]:
contracts_raw = contracts.copy()

# Pequena imperfeição: alguns contratos sem status informado
missing_status_idx = contracts_raw.sample(frac=0.01, random_state=RANDOM_SEED + 3).index
contracts_raw.loc[missing_status_idx, "contract_status"] = ""

# Registro duplicado proposital
contracts_raw = pd.concat(
    [contracts_raw, contracts_raw.sample(n=1, random_state=RANDOM_SEED + 4)],
    ignore_index=True,
)

contracts_raw.head()

,contract_id,customer_id,product_type,principal_amount,interest_rate_monthly,term_months,start_date,contract_status,source_system,extraction_date
0,CTR000001,C000001,credit_card,2863.83,0.0844,24,2022-12-05,active,loan_management,2026-04-28
1,CTR000002,C000002,auto_loan,3431.72,0.0225,48,2022-01-01,closed,loan_management,2026-04-28
2,CTR000003,C000002,credit_card,232.56,0.0538,24,2022-12-23,active,loan_management,2026-04-28
3,CTR000004,C000003,credit_card,4453.18,0.0637,24,2020-06-05,closed,loan_management,2026-04-28
4,CTR000005,C000003,payroll_loan,15677.32,0.0193,24,2021-03-20,,loan_management,2026-04-28


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 5. Geração da entidade "payments"</strong></font>

<font size="2" color='#66666'>

>- A tabela `payments` representa o histórico mensal de pagamentos dos contratos.

>- Ela inclui:
    - número da parcela;
    - data de vencimento;
    - data de pagamento;
    - valor esperado;
    - valor pago;
    - dias de atraso.

>- Esta entidade será essencial para criar variáveis de risco na camada Gold.

---

</font></div>

In [12]:
payment_rows = []
payment_counter = 1

contracts_enriched = contracts.merge(
    customers[["customer_id", "risk_band_source", "credit_score_base"]],
    on="customer_id",
    how="left",
)

for _, contract in contracts_enriched.iterrows():
    months_to_generate = min(contract["term_months"], MAX_PAYMENT_MONTHS)
    start_dt = pd.to_datetime(contract["start_date"]).date()
    monthly_amount = round(
        contract["principal_amount"] 
        * (1 + contract["interest_rate_monthly"] * contract["term_months"])
        / contract["term_months"],
        2,
    )
    
    risk_band = contract["risk_band_source"]
    
    late_probability = {
        "low": 0.08,
        "medium": 0.18,
        "high": 0.34,
    }[risk_band]
    
    for installment in range(1, months_to_generate + 1):
        due_dt = start_dt + timedelta(days=30 * installment)
        
        is_late = np.random.random() < late_probability
        is_unpaid = np.random.random() < (late_probability * 0.12)
        
        if is_unpaid:
            payment_dt = ""
            days_late = int(np.random.choice([30, 45, 60, 90, 120]))
            amount_paid = 0.0
            payment_status = "unpaid"
        elif is_late:
            days_late = int(np.random.choice([5, 10, 15, 30, 45, 60]))
            payment_dt = (due_dt + timedelta(days=days_late)).isoformat()
            amount_paid = monthly_amount
            payment_status = "late"
        else:
            days_late = int(np.random.choice([-5, -2, 0, 0, 0, 1]))
            payment_dt = (due_dt + timedelta(days=days_late)).isoformat()
            amount_paid = monthly_amount
            payment_status = "paid"
        
        payment_rows.append(
            {
                "payment_id": make_id("PAY", payment_counter),
                "contract_id": contract["contract_id"],
                "installment_number": installment,
                "due_date": due_dt.isoformat(),
                "payment_date": payment_dt,
                "scheduled_amount": monthly_amount,
                "amount_paid": round(amount_paid, 2),
                "days_late": days_late,
                "payment_status": payment_status,
                "source_system": "payment_gateway",
                "extraction_date": STUDY_DATE,
            }
        )
        
        payment_counter += 1

payments = pd.DataFrame(payment_rows)

payments.head()


,payment_id,contract_id,installment_number,due_date,payment_date,scheduled_amount,amount_paid,days_late,payment_status,source_system,extraction_date
0,PAY000001,CTR000001,1,2023-01-04,2023-02-03,361.03,361.03,30,late,payment_gateway,2026-04-28
1,PAY000002,CTR000001,2,2023-02-03,2023-02-03,361.03,361.03,0,paid,payment_gateway,2026-04-28
2,PAY000003,CTR000001,3,2023-03-05,2023-03-06,361.03,361.03,1,paid,payment_gateway,2026-04-28
3,PAY000004,CTR000001,4,2023-04-04,2023-06-03,361.03,361.03,60,late,payment_gateway,2026-04-28
4,PAY000005,CTR000001,5,2023-05-04,2023-05-05,361.03,361.03,1,paid,payment_gateway,2026-04-28


In [13]:
payments_raw = payments.copy()

# Pequena imperfeição: alguns amount_paid ausentes
missing_amount_idx = payments_raw.sample(frac=0.005, random_state=RANDOM_SEED + 5).index
payments_raw.loc[missing_amount_idx, "amount_paid"] = np.nan

# Pequena imperfeição: 1 pagamento duplicado
payments_raw = pd.concat(
    [payments_raw, payments_raw.sample(n=1, random_state=RANDOM_SEED + 6)],
    ignore_index=True,
)

payments_raw.head()

,payment_id,contract_id,installment_number,due_date,payment_date,scheduled_amount,amount_paid,days_late,payment_status,source_system,extraction_date
0,PAY000001,CTR000001,1,2023-01-04,2023-02-03,361.03,361.03,30,late,payment_gateway,2026-04-28
1,PAY000002,CTR000001,2,2023-02-03,2023-02-03,361.03,361.03,0,paid,payment_gateway,2026-04-28
2,PAY000003,CTR000001,3,2023-03-05,2023-03-06,361.03,361.03,1,paid,payment_gateway,2026-04-28
3,PAY000004,CTR000001,4,2023-04-04,2023-06-03,361.03,361.03,60,late,payment_gateway,2026-04-28
4,PAY000005,CTR000001,5,2023-05-04,2023-05-05,361.03,361.03,1,paid,payment_gateway,2026-04-28


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 6. Geração da entidade "credit_events"</strong></font>

<font size="2" color='#66666'>

>- A tabela `credit_events` representa eventos relevantes no ciclo de crédito.

>- Exemplos:
    - atualização cadastral;
    - renegociação;
    - contato de cobrança;
    - alteração de limite;
    - alerta de atraso;
    - atualização de bureau.

>- Esses eventos ajudam a enriquecer o perfil do cliente e podem ser usados futuramente em análises comportamentais.

---

</font></div>

In [14]:
event_types = [
    "bureau_update",
    "collection_contact",
    "renegotiation",
    "limit_change",
    "delinquency_alert",
    "customer_update",
]

event_probs = [0.25, 0.18, 0.10, 0.16, 0.16, 0.15]

event_rows = []
event_counter = 1

contracts_sample = contracts[["contract_id", "customer_id", "start_date"]].copy()

for _, contract in contracts_sample.iterrows():
    n_events = np.random.choice([0, 1, 2, 3], p=[0.38, 0.36, 0.18, 0.08])
    contract_start = pd.to_datetime(contract["start_date"]).date()
    
    for _ in range(n_events):
        event_type = random_choice_with_probs(event_types, event_probs)
        event_dt = random_date(contract_start, date(2025, 12, 31))
        
        event_value = {
            "bureau_update": np.random.randint(300, 900),
            "collection_contact": np.random.choice([1, 2, 3, 4]),
            "renegotiation": round(np.random.uniform(500, 15000), 2),
            "limit_change": round(np.random.uniform(-3000, 8000), 2),
            "delinquency_alert": np.random.choice([15, 30, 60, 90]),
            "customer_update": 1,
        }[event_type]
        
        event_rows.append(
            {
                "event_id": make_id("EVT", event_counter),
                "customer_id": contract["customer_id"],
                "contract_id": contract["contract_id"],
                "event_type": event_type,
                "event_date": event_dt.isoformat(),
                "event_value": event_value,
                "source_system": "credit_operations",
                "extraction_date": STUDY_DATE,
            }
        )
        
        event_counter += 1

credit_events = pd.DataFrame(event_rows)

credit_events.head()


,event_id,customer_id,contract_id,event_type,event_date,event_value,source_system,extraction_date
0,EVT000001,C000001,CTR000001,collection_contact,2024-03-27,1.0,credit_operations,2026-04-28
1,EVT000002,C000001,CTR000001,bureau_update,2025-07-27,886.0,credit_operations,2026-04-28
2,EVT000003,C000002,CTR000002,delinquency_alert,2022-02-21,30.0,credit_operations,2026-04-28
3,EVT000004,C000002,CTR000002,collection_contact,2025-06-22,2.0,credit_operations,2026-04-28
4,EVT000005,C000002,CTR000003,bureau_update,2024-10-24,444.0,credit_operations,2026-04-28


In [15]:
credit_events_raw = credit_events.copy()

# Pequena imperfeição: alguns event_value ausentes
if len(credit_events_raw) > 0:
    missing_event_value_idx = credit_events_raw.sample(frac=0.01, random_state=RANDOM_SEED + 7).index
    credit_events_raw.loc[missing_event_value_idx, "event_value"] = np.nan

credit_events_raw.head()

,event_id,customer_id,contract_id,event_type,event_date,event_value,source_system,extraction_date
0,EVT000001,C000001,CTR000001,collection_contact,2024-03-27,1.0,credit_operations,2026-04-28
1,EVT000002,C000001,CTR000001,bureau_update,2025-07-27,886.0,credit_operations,2026-04-28
2,EVT000003,C000002,CTR000002,delinquency_alert,2022-02-21,30.0,credit_operations,2026-04-28
3,EVT000004,C000002,CTR000002,collection_contact,2025-06-22,2.0,credit_operations,2026-04-28
4,EVT000005,C000002,CTR000003,bureau_update,2024-10-24,444.0,credit_operations,2026-04-28


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 7. Visão geral dos dados gerados</strong></font>

<font size="2" color='#66666'>

>- Antes de salvar os arquivos Raw, vamos verificar o volume de registros por entidade.

---

</font></div>

In [16]:
raw_tables = {
    "customers": customers_raw,
    "contracts": contracts_raw,
    "payments": payments_raw,
    "credit_events": credit_events_raw,
}

raw_volume_summary = pd.DataFrame(
    [
        {
            "entity": entity,
            "rows": len(df),
            "columns": len(df.columns),
        }
        for entity, df in raw_tables.items()
    ]
)

raw_volume_summary

,entity,rows,columns
0,customers,501,11
1,contracts,622,10
2,payments,12919,11
3,credit_events,580,8


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 8. Perfil inicial de qualidade dos arquivos Raw</strong></font>

<font size="2" color='#66666'>

>- A camada Raw não corrige os problemas.

>- Mesmo assim, é útil gerar um perfil inicial para registrar:
    - quantidade de linhas;
    - quantidade de colunas;
    - células nulas;
    - linhas duplicadas;
    - percentual de completude.

>- Esse perfil será usado como insumo para as camadas Bronze e Silver.

---

</font></div>

In [17]:
def profile_raw_frame(entity_name: str, df: pd.DataFrame) -> pd.DataFrame:
    """
    Gera um perfil simples de qualidade por coluna.
    """
    rows = []
    total_rows = len(df)
    
    for column in df.columns:
        null_count = df[column].isna().sum() + (df[column].astype(str).str.strip() == "").sum()
        null_pct = round(null_count / total_rows, 4) if total_rows > 0 else 0
        
        rows.append(
            {
                "entity": entity_name,
                "column": column,
                "total_rows": total_rows,
                "null_or_blank_count": int(null_count),
                "null_or_blank_pct": null_pct,
                "inferred_dtype": str(df[column].dtype),
                "sample_values": ", ".join(df[column].dropna().astype(str).head(3).tolist()),
            }
        )
    
    return pd.DataFrame(rows)


raw_profile_summary = pd.concat(
    [
        profile_raw_frame(entity, df)
        for entity, df in raw_tables.items()
    ],
    ignore_index=True,
)

raw_profile_summary.head(20)


,entity,column,total_rows,null_or_blank_count,null_or_blank_pct,inferred_dtype,sample_values
0,customers,customer_id,501,0,0.0000,str,"C000001, C000002, C000003"
1,customers,customer_name,501,0,0.0000,str,"Customer 0001, Customer 0002, Customer 0003"
2,customers,age,501,0,0.0000,int64,"46, 34, 58"
3,customers,monthly_income,501,15,0.0299,float64,"7501.58, 500.0, 4725.87"
4,customers,employment_type,501,0,0.0000,str,"public_servant, formal_employee, formal_employee"
5,customers,region,501,0,0.0000,str,"SP, RJ, SP"
6,customers,credit_score_base,501,0,0.0000,int64,"594, 579, 592"
7,customers,risk_band_source,501,0,0.0000,str,"high, high, high"
8,customers,signup_date,501,0,0.0000,str,"2025-03-04, 2019-04-02, 2018-04-13"
9,customers,source_system,501,0,0.0000,str,"crm_core, crm_core, crm_core"


In [18]:
raw_entity_quality_summary = pd.DataFrame(
    [
        {
            "entity": entity,
            "rows": len(df),
            "columns": len(df.columns),
            "duplicated_rows": int(df.duplicated().sum()),
            "total_null_cells": int(df.isna().sum().sum()),
            "total_blank_cells": int((df.astype(str).apply(lambda col: col.str.strip() == "")).sum().sum()),
        }
        for entity, df in raw_tables.items()
    ]
)

raw_entity_quality_summary

,entity,rows,columns,duplicated_rows,total_null_cells,total_blank_cells
0,customers,501,11,1,15,0
1,contracts,622,10,1,0,6
2,payments,12919,11,1,65,313
3,credit_events,580,8,0,6,0


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 9. Dicionário de dados inicial</strong></font>

<font size="2" color='#66666'>

>- O dicionário de dados descreve o significado das entidades e colunas.

>- Neste momento, ele representa a expectativa inicial de schema para os arquivos Raw.

---

</font></div>

In [19]:
data_dictionary_rows = [
    # customers
    ("customers", "customer_id", "Identificador único do cliente.", "string", "primary_key", "no"),
    ("customers", "customer_name", "Nome sintético do cliente.", "string", "attribute", "no"),
    ("customers", "age", "Idade do cliente.", "integer", "attribute", "no"),
    ("customers", "monthly_income", "Renda mensal declarada.", "float", "attribute", "yes"),
    ("customers", "employment_type", "Tipo de ocupação.", "string", "attribute", "no"),
    ("customers", "region", "Unidade federativa/região sintética.", "string", "attribute", "no"),
    ("customers", "credit_score_base", "Score de crédito sintético inicial.", "integer", "attribute", "no"),
    ("customers", "risk_band_source", "Faixa de risco derivada do score inicial.", "string", "attribute", "no"),
    ("customers", "signup_date", "Data de cadastro.", "date", "attribute", "no"),
    
    # contracts
    ("contracts", "contract_id", "Identificador único do contrato.", "string", "primary_key", "no"),
    ("contracts", "customer_id", "Identificador do cliente relacionado.", "string", "foreign_key", "no"),
    ("contracts", "product_type", "Tipo de produto de crédito.", "string", "attribute", "no"),
    ("contracts", "principal_amount", "Valor principal do contrato.", "float", "measure", "no"),
    ("contracts", "interest_rate_monthly", "Taxa de juros mensal.", "float", "measure", "no"),
    ("contracts", "term_months", "Prazo do contrato em meses.", "integer", "attribute", "no"),
    ("contracts", "start_date", "Data de início do contrato.", "date", "attribute", "no"),
    ("contracts", "contract_status", "Status do contrato.", "string", "attribute", "yes"),
    
    # payments
    ("payments", "payment_id", "Identificador único do pagamento.", "string", "primary_key", "no"),
    ("payments", "contract_id", "Identificador do contrato relacionado.", "string", "foreign_key", "no"),
    ("payments", "installment_number", "Número da parcela.", "integer", "attribute", "no"),
    ("payments", "due_date", "Data de vencimento.", "date", "attribute", "no"),
    ("payments", "payment_date", "Data de pagamento.", "date", "attribute", "yes"),
    ("payments", "scheduled_amount", "Valor esperado da parcela.", "float", "measure", "no"),
    ("payments", "amount_paid", "Valor pago.", "float", "measure", "yes"),
    ("payments", "days_late", "Dias de atraso.", "integer", "measure", "no"),
    ("payments", "payment_status", "Status do pagamento.", "string", "attribute", "no"),
    
    # credit_events
    ("credit_events", "event_id", "Identificador único do evento.", "string", "primary_key", "no"),
    ("credit_events", "customer_id", "Identificador do cliente relacionado.", "string", "foreign_key", "no"),
    ("credit_events", "contract_id", "Identificador do contrato relacionado.", "string", "foreign_key", "no"),
    ("credit_events", "event_type", "Tipo de evento de crédito.", "string", "attribute", "no"),
    ("credit_events", "event_date", "Data do evento.", "date", "attribute", "no"),
    ("credit_events", "event_value", "Valor associado ao evento.", "float/string", "measure", "yes"),
]

raw_data_dictionary = pd.DataFrame(
    data_dictionary_rows,
    columns=[
        "entity",
        "column",
        "description",
        "expected_type",
        "column_role",
        "nullable_expected",
    ],
)

# Colunas técnicas comuns
technical_columns = pd.DataFrame(
    [
        {
            "entity": "all",
            "column": "source_system",
            "description": "Sistema de origem do registro.",
            "expected_type": "string",
            "column_role": "metadata",
            "nullable_expected": "no",
        },
        {
            "entity": "all",
            "column": "extraction_date",
            "description": "Data de extração do registro.",
            "expected_type": "date",
            "column_role": "metadata",
            "nullable_expected": "no",
        },
    ]
)

raw_data_dictionary = pd.concat(
    [raw_data_dictionary, technical_columns],
    ignore_index=True,
)

raw_data_dictionary.head(20)


,entity,column,description,expected_type,column_role,nullable_expected
0,customers,customer_id,Identificador único do cliente.,string,primary_key,no
1,customers,customer_name,Nome sintético do cliente.,string,attribute,no
2,customers,age,Idade do cliente.,integer,attribute,no
3,customers,monthly_income,Renda mensal declarada.,float,attribute,yes
4,customers,employment_type,Tipo de ocupação.,string,attribute,no
5,customers,region,Unidade federativa/região sintética.,string,attribute,no
6,customers,credit_score_base,Score de crédito sintético inicial.,integer,attribute,no
7,customers,risk_band_source,Faixa de risco derivada do score inicial.,string,attribute,no
8,customers,signup_date,Data de cadastro.,date,attribute,no
9,contracts,contract_id,Identificador único do contrato.,string,primary_key,no


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 10. Salvando os arquivos na camada Raw</strong></font>

<font size="2" color='#66666'>

>- Agora os arquivos serão salvos em `data/raw`.

>- Eles serão usados no próximo notebook para iniciar a ingestão Raw → Bronze.

---

</font></div>

In [20]:
raw_file_paths = {
    "customers": RAW_DIR / "customers.csv",
    "contracts": RAW_DIR / "contracts.csv",
    "payments": RAW_DIR / "payments.csv",
    "credit_events": RAW_DIR / "credit_events.csv",
}

for entity, path in raw_file_paths.items():
    safe_to_csv(raw_tables[entity], path)

sorted([path.name for path in RAW_DIR.glob("*.csv")])

['contracts.csv', 'credit_events.csv', 'customers.csv', 'payments.csv']

In [21]:
raw_layer_manifest = pd.DataFrame(
    [
        {
            "entity": entity,
            "file_name": path.name,
            "file_path": str(path),
            "layer": "raw",
            "file_format": "csv",
            "rows": len(raw_tables[entity]),
            "columns": len(raw_tables[entity].columns),
            "source_systems": ", ".join(sorted(raw_tables[entity]["source_system"].dropna().astype(str).unique())),
            "generation_date": STUDY_DATE,
            "notebook": NOTEBOOK_NAME,
        }
        for entity, path in raw_file_paths.items()
    ]
)

raw_layer_manifest


,entity,file_name,file_path,layer,file_format,rows,columns,source_systems,generation_date,notebook
0,customers,customers.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\customers.csv,raw,csv,501,11,crm_core,2026-04-28,02_data_source_and_raw_layer_preparation
1,contracts,contracts.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\contracts.csv,raw,csv,622,10,loan_management,2026-04-28,02_data_source_and_raw_layer_preparation
2,payments,payments.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\payments.csv,raw,csv,12919,11,payment_gateway,2026-04-28,02_data_source_and_raw_layer_preparation
3,credit_events,credit_events.csv,D:\_DS-Projects\Data-Science\databricks-credit-risk-lakehouse\data\raw\credit_events.csv,raw,csv,580,8,credit_operations,2026-04-28,02_data_source_and_raw_layer_preparation


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 11. Evidências de aprendizagem para o KG</strong></font>

<font size="2" color='#66666'>

>- Este notebook gera evidência real para o `databricks-learning-kg`.

>- A evidência agora é mais prática do que no Notebook 01, porque criamos dados, schemas, arquivos, manifesto e perfil de qualidade inicial.

---

</font></div>

In [22]:
learning_evidence = pd.DataFrame(
    [
        {
            "evidence_id": "EVID_CR_LH_007",
            "project": PROJECT_NAME,
            "notebook": NOTEBOOK_NAME,
            "evidence_date": STUDY_DATE,
            "topic": "Data Ingestion",
            "evidence_type": "raw_layer_preparation",
            "confidence_delta_suggested": 0.15,
            "mastery_delta_suggested": 0.05,
            "evidence_description": "Preparação dos arquivos brutos que serão usados como origem para ingestão Raw to Bronze.",
        },
        {
            "evidence_id": "EVID_CR_LH_008",
            "project": PROJECT_NAME,
            "notebook": NOTEBOOK_NAME,
            "evidence_date": STUDY_DATE,
            "topic": "File Formats",
            "evidence_type": "csv_generation",
            "confidence_delta_suggested": 0.10,
            "mastery_delta_suggested": 0.05,
            "evidence_description": "Geração de arquivos CSV para customers, contracts, payments e credit_events.",
        },
        {
            "evidence_id": "EVID_CR_LH_009",
            "project": PROJECT_NAME,
            "notebook": NOTEBOOK_NAME,
            "evidence_date": STUDY_DATE,
            "topic": "Schema Definition",
            "evidence_type": "schema_documentation",
            "confidence_delta_suggested": 0.15,
            "mastery_delta_suggested": 0.05,
            "evidence_description": "Criação de dicionário de dados inicial com tipos esperados, papéis de coluna e nulidade esperada.",
        },
        {
            "evidence_id": "EVID_CR_LH_010",
            "project": PROJECT_NAME,
            "notebook": NOTEBOOK_NAME,
            "evidence_date": STUDY_DATE,
            "topic": "Data Quality",
            "evidence_type": "raw_quality_profile",
            "confidence_delta_suggested": 0.10,
            "mastery_delta_suggested": 0.05,
            "evidence_description": "Geração de perfil inicial de qualidade com nulos, blanks, duplicidades e tipos inferidos.",
        },
        {
            "evidence_id": "EVID_CR_LH_011",
            "project": PROJECT_NAME,
            "notebook": NOTEBOOK_NAME,
            "evidence_date": STUDY_DATE,
            "topic": "Naming Conventions",
            "evidence_type": "raw_file_naming",
            "confidence_delta_suggested": 0.05,
            "mastery_delta_suggested": 0.05,
            "evidence_description": "Aplicação de convenções simples de nomes para entidades e arquivos Raw.",
        },
    ]
)

learning_evidence

,evidence_id,project,notebook,evidence_date,topic,evidence_type,confidence_delta_suggested,mastery_delta_suggested,evidence_description
0,EVID_CR_LH_007,Databricks Credit Risk Lakehouse,02_data_source_and_raw_layer_preparation,2026-04-28,Data Ingestion,raw_layer_preparation,0.15,0.05,Preparação dos arquivos brutos que serão usados como origem para ingestão Raw to Bronze.
1,EVID_CR_LH_008,Databricks Credit Risk Lakehouse,02_data_source_and_raw_layer_preparation,2026-04-28,File Formats,csv_generation,0.10,0.05,"Geração de arquivos CSV para customers, contracts, payments e credit_events."
2,EVID_CR_LH_009,Databricks Credit Risk Lakehouse,02_data_source_and_raw_layer_preparation,2026-04-28,Schema Definition,schema_documentation,0.15,0.05,"Criação de dicionário de dados inicial com tipos esperados, papéis de coluna e nulidade esperada."
3,EVID_CR_LH_010,Databricks Credit Risk Lakehouse,02_data_source_and_raw_layer_preparation,2026-04-28,Data Quality,raw_quality_profile,0.10,0.05,"Geração de perfil inicial de qualidade com nulos, blanks, duplicidades e tipos inferidos."
4,EVID_CR_LH_011,Databricks Credit Risk Lakehouse,02_data_source_and_raw_layer_preparation,2026-04-28,Naming Conventions,raw_file_naming,0.05,0.05,Aplicação de convenções simples de nomes para entidades e arquivos Raw.


<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 12. Exportação dos artefatos</strong></font>

<font size="2" color='#66666'>

>- Nesta etapa, salvamos os arquivos auxiliares em `data/exports` e `docs`.

>- Esses artefatos servirão para documentação, auditoria e atualização futura do Knowledge Graph.

---

</font></div>

In [23]:
safe_to_csv(raw_volume_summary, EXPORTS_DIR / "raw_volume_summary.csv")
safe_to_csv(raw_profile_summary, EXPORTS_DIR / "raw_profile_summary.csv")
safe_to_csv(raw_entity_quality_summary, EXPORTS_DIR / "raw_entity_quality_summary.csv")
safe_to_csv(raw_data_dictionary, EXPORTS_DIR / "raw_data_dictionary.csv")
safe_to_csv(raw_layer_manifest, EXPORTS_DIR / "raw_layer_manifest.csv")
safe_to_csv(learning_evidence, EXPORTS_DIR / "learning_evidence_notebook_02.csv")

print("Artefatos exportados com sucesso.")

Artefatos exportados com sucesso.


In [24]:
summary_report = f"""# Notebook 02 — Data Source and Raw Layer Preparation

## Projeto

{PROJECT_NAME}

## Data

{STUDY_DATE}

## Objetivo

Criar a base inicial de dados sintéticos para o projeto de risco de crédito e preparar os arquivos da camada Raw.

## Entidades geradas

{raw_volume_summary.to_markdown(index=False)}

## Manifesto da camada Raw

{raw_layer_manifest[["entity", "file_name", "layer", "file_format", "rows", "columns", "source_systems"]].to_markdown(index=False)}

## Qualidade inicial por entidade

{raw_entity_quality_summary.to_markdown(index=False)}

## Evidências de aprendizagem geradas

{learning_evidence[["evidence_id", "topic", "evidence_type", "confidence_delta_suggested", "mastery_delta_suggested"]].to_markdown(index=False)}

## Leitura executiva

Este notebook criou a primeira fonte de dados do projeto `Databricks Credit Risk Lakehouse`.

Foram gerados arquivos brutos para clientes, contratos, pagamentos e eventos de crédito. Os dados incluem pequenas imperfeições controladas, como valores ausentes, blanks e duplicidades, para permitir a prática realista de ingestão, validação e tratamento nas próximas camadas.

## Próximo passo

O próximo notebook será:

`Notebook 03 — Raw to Bronze Ingestion`

Nele, os arquivos CSV da camada Raw serão lidos, enriquecidos com metadados técnicos e persistidos na camada Bronze em formato analítico.
"""

report_path = DOCS_DIR / "notebook_02_data_source_and_raw_layer_summary.md"
report_path.write_text(summary_report, encoding="utf-8")

report_path

WindowsPath('D:/_DS-Projects/Data-Science/databricks-credit-risk-lakehouse/docs/notebook_02_data_source_and_raw_layer_summary.md')

<div style="background-color:#f3f2ee">
    
<font size="4" color='#CC403E'><strong>✔️ 13. Conclusão executiva</strong></font>

<font size="2" color='#66666'>

>- Este notebook preparou a fonte de dados inicial do projeto `Databricks Credit Risk Lakehouse`.

>- Foram criadas quatro entidades principais:
    - customers;
    - contracts;
    - payments;
    - credit_events.

>- Também foram gerados:
    - arquivos CSV na camada Raw;
    - manifesto da camada Raw;
    - dicionário de dados inicial;
    - perfil de qualidade inicial;
    - sumário de volume;
    - evidências reais para atualização futura do Knowledge Graph.

>- **Leitura executiva**
    - A camada Raw agora possui dados suficientes para iniciar uma esteira Lakehouse progressiva.
    - Os arquivos gerados preservam características típicas de dados brutos, incluindo pequenas inconsistências,
        - permitindo que os próximos notebooks implementem ingestão, validação, padronização e tratamento.

>- **Próximo notebook**
    - `Notebook 03 — Raw to Bronze Ingestion`
    - O próximo passo será transformar os arquivos Raw em tabelas Bronze com metadados de ingestão, rastreabilidade e persistência em formato analítico.

---

</font></div>

In [25]:
%reload_ext watermark 
%watermark -a "Roberto-SSoares-LfLngLrnng" -d -t -u --iversions -v -m -h

Author: Roberto-SSoares-LfLngLrnng

Last updated: 2026-04-28 19:51:15

Python implementation: CPython
Python version       : 3.12.12
IPython version      : 9.13.0

Compiler    : MSC v.1944 64 bit (AMD64)
OS          : Windows
Release     : 11
Machine     : AMD64
Processor   : Intel64 Family 6 Model 158 Stepping 9, GenuineIntel
CPU cores   : 4
Architecture: 64bit

Hostname: PC-ROBERTO

numpy : 2.4.4
pandas: 3.0.2



<div style="background-color:#f3f2ee">
    
<font size="6" color='#CC403E'><strong>Fim</strong></font>

<font size="3" color='#66666'></font></div>


In [26]:
#!uv pip install nbconvert -U -q
!jupyter nbconvert --to html --template-file my-template-html-v10.tpl 02_data_source_and_raw_layer_preparation.ipynb

[NbConvertApp] Converting notebook 02_data_source_and_raw_layer_preparation.ipynb to html
[NbConvertApp] Writing 82948 bytes to 02_data_source_and_raw_layer_preparation.html
